### Start Spark Session

In [6]:
import pyspark 
print(pyspark.__version__)

4.1.1


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkTutorial") \
    .getOrCreate()

spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 15:20:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



### Test creating RDD

In [8]:
spark = SparkSession.builder.getOrCreate()

sc = spark.sparkContext

rdd = sc.parallelize([1, 2, 3, 4, 5])
rdd.collect()

[1, 2, 3, 4, 5]

In [9]:
rdd2 = rdd.map(lambda x: x * 10)   # no computation yet
rdd2.take(3)  

[10, 20, 30]

## ML example with a real dataset
For most ML algorithm within PySpark, they are not classic MapReduce.

Classic MapReduce means you structure the whole job as:
```
map → shuffle/group-by key → reduce
(one or a small fixed number of stages; batch; heavy “shuffle then reduce” pattern)
```

Machine learning models are typically done with an iterative optimization algorithm (e.g., gradient-based methods or solvers that repeatedly update parameters). That workflow looks more like:
```
repeat until convergence:
compute gradients/statistics over data in parallel → aggregate → update weights
```

Spark executes this as a DAG of transformations and actions, possibly with caching, and can run many stages across iterations.

In [10]:
from typing import List, Tuple

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler

In [11]:
# ----------------------------
# 1) Spark session
# ----------------------------
def start_spark(app_name: str = "LD2011_2014_pipeline") -> SparkSession:
    """
    Start (or get) a SparkSession. Adjust configs here if needed.
    """
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")   # optional: comment out if you're on a cluster
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [12]:
# ----------------------------
# 2) Read + parse wide format
# ----------------------------
def read_ld_wide(
    spark: SparkSession,
    raw_path: str,
    sep: str = ";",
) -> DataFrame:
    """
    Read the UCI LD2011_2014.txt as a wide CSV-like file.
    """
    return (
        spark.read
        .option("sep", sep)
        .option("header", False)
        .option("inferSchema", False)
        .csv(raw_path)
    )

In [15]:
def rename_and_parse_timestamp(df_wide: DataFrame) -> DataFrame:
    """
    Rename columns to ts_raw + MT_001..MT_370 and parse timestamp.
    """
    n = len(df_wide.columns)  # should be 371 for LD2011_2014
    new_cols = ["ts_raw"] + [f"MT_{i:03d}" for i in range(1, n)]
    df_wide = df_wide.toDF(*new_cols)

    df = (
        df_wide
        .withColumn("ts", F.to_timestamp("ts_raw", "yyyy-MM-dd HH:mm:ss"))
        .drop("ts_raw")
        .dropna(subset=["ts"])
    )
    return df

In [16]:
def cast_meter_columns_to_double(df: DataFrame) -> Tuple[DataFrame, List[str]]:
    """
    Convert meter columns to double, handling comma decimals. Returns (df, meter_cols).
    """
    meter_cols = [c for c in df.columns if c != "ts"]
    for c in meter_cols:
        df = df.withColumn(
            c,
            F.expr(f"try_cast(regexp_replace(`{c}`, ',', '.') as double)")
        )
    return df, meter_cols

In [17]:
# ----------------------------
# 3) Wide -> long
# ----------------------------
def wide_to_long(df: DataFrame, meter_cols: List[str]) -> DataFrame:
    """
    Convert wide meter columns into long format: (ts, client_id, y).
    """
    stack_expr = ", ".join([f"'{c}', `{c}`" for c in meter_cols])
    df_long = (
        df.selectExpr(
            "ts",
            f"stack({len(meter_cols)}, {stack_expr}) as (client_id, y)"
        )
        .dropna(subset=["ts", "y"])
    )
    return df_long

In [18]:
def sample_clients(df_long: DataFrame, k: int = 3, seed: int = 42) -> List[str]:
    """
    Pick k distinct clients. Uses deterministic order; you can also randomize if you want.
    """
    # Deterministic: orderBy client_id then take k
    rows = (
        df_long.select("client_id")
        .distinct()
        .orderBy("client_id")
        .limit(k)
        .collect()
    )
    return [r["client_id"] for r in rows]

In [19]:
def filter_clients(df_long: DataFrame, clients: List[str]) -> DataFrame:
    return df_long.filter(F.col("client_id").isin(clients))# ----------------------------
# 4) Feature engineering (window)
# ----------------------------
def build_features(df_small: DataFrame) -> DataFrame:
    """
    Create lag/rolling/time features and label using lead().
    """
    w = Window.partitionBy("client_id").orderBy("ts")

    df_feat = (
        df_small
        .withColumn("y_next", F.lead("y", 1).over(w))     # label
        .withColumn("lag1",  F.lag("y", 1).over(w))
        .withColumn("lag4",  F.lag("y", 4).over(w))
        .withColumn("lag96", F.lag("y", 96).over(w))
        .withColumn("roll4_mean",  F.avg("y").over(w.rowsBetween(-4, -1)))
        .withColumn("roll96_mean", F.avg("y").over(w.rowsBetween(-96, -1)))
        .withColumn("hour", F.hour("ts"))
        .withColumn("dow",  F.dayofweek("ts"))
        .dropna(subset=["y_next", "lag1", "lag4", "lag96", "roll4_mean", "roll96_mean"])
        .withColumnRenamed("y_next", "label")
    )
    return df_feat

In [20]:
def add_baselines(df_feat: DataFrame) -> DataFrame:
    """
    Add naive and seasonal baseline predictions.
    """
    return (
        df_feat
        .withColumn("pred_naive", F.col("lag1"))
        .withColumn("pred_seasonal", F.col("lag96"))
    )

In [21]:
# ----------------------------
# 5) Train/test split + assemble
# ----------------------------
def train_test_split_by_ts(df: DataFrame, cutoff: str = "2014-01-01 00:00:00") -> Tuple[DataFrame, DataFrame]:
    cutoff_ts = F.to_timestamp(F.lit(cutoff))
    train = df.filter(F.col("ts") < cutoff_ts)
    test  = df.filter(F.col("ts") >= cutoff_ts)
    return train, test

In [22]:
def assemble_features(
    train: DataFrame,
    test: DataFrame,
    feature_cols: List[str],
    features_col: str = "features"
) -> Tuple[DataFrame, DataFrame]:
    assembler = VectorAssembler(inputCols=feature_cols, outputCol=features_col)
    train_ml = assembler.transform(train).select("ts", "client_id", features_col, "label")
    test_ml  = assembler.transform(test).select("ts", "client_id", features_col, "label")
    return train_ml, test_ml

In [23]:
def run_pipeline(raw_path: str, k_clients: int = 3) -> Tuple[DataFrame, DataFrame]:
    spark = start_spark()

    df_wide = read_ld_wide(spark, raw_path)
    df = rename_and_parse_timestamp(df_wide)
    df, meter_cols = cast_meter_columns_to_double(df)

    # (optional sanity check)
    df.select("ts", meter_cols[0]).show(5, truncate=False)

    df_long = wide_to_long(df, meter_cols)
    df_long.show(5, truncate=False)

    clients = sample_clients(df_long, k=k_clients)
    df_small = filter_clients(df_long, clients)

    df_feat = build_features(df_small)
    df_base = add_baselines(df_feat)

    train, test = train_test_split_by_ts(df_base, cutoff="2012-06-01 00:00:00")

    feature_cols = ["lag1", "lag4", "lag96", "roll4_mean", "roll96_mean", "hour", "dow"]
    train_ml, test_ml = assemble_features(train, test, feature_cols)

    return train_ml, test_ml

In [24]:
import os, shutil

src = f"LD2011_2014.txt"
dst = f"/tmp/LD2011_2014.txt"

shutil.copyfile(src, dst)
raw_path = os.path.abspath(dst)

train_ml, test_ml = run_pipeline(raw_path, k_clients=3)

train_ml.show(5, truncate=False)
test_ml.show(5, truncate=False)

26/05/12 17:31:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------------------+------+
|ts                 |MT_001|
+-------------------+------+
|2011-01-01 00:15:00|0.0   |
|2011-01-01 00:30:00|0.0   |
|2011-01-01 00:45:00|0.0   |
|2011-01-01 01:00:00|0.0   |
|2011-01-01 01:15:00|0.0   |
+-------------------+------+
only showing top 5 rows


26/05/12 17:32:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+---------+---+
|ts                 |client_id|y  |
+-------------------+---------+---+
|2011-01-01 00:15:00|MT_001   |0.0|
|2011-01-01 00:15:00|MT_002   |0.0|
|2011-01-01 00:15:00|MT_003   |0.0|
|2011-01-01 00:15:00|MT_004   |0.0|
|2011-01-01 00:15:00|MT_005   |0.0|
+-------------------+---------+---+
only showing top 5 rows


+-------------------+---------+-------------------+-----+
|ts                 |client_id|features           |label|
+-------------------+---------+-------------------+-----+
|2011-01-02 00:15:00|MT_002   |(7,[6],[1.0])      |0.0  |
|2011-01-02 00:30:00|MT_002   |(7,[6],[1.0])      |0.0  |
|2011-01-02 00:45:00|MT_002   |(7,[6],[1.0])      |0.0  |
|2011-01-02 01:00:00|MT_002   |(7,[5,6],[1.0,1.0])|0.0  |
|2011-01-02 01:15:00|MT_002   |(7,[5,6],[1.0,1.0])|0.0  |
+-------------------+---------+-------------------+-----+
only showing top 5 rows


+-------------------+---------+--------------------------------------------------------------------------------------------------+----------------+
|ts                 |client_id|features                                                                                          |label           |
+-------------------+---------+--------------------------------------------------------------------------------------------------+----------------+
|2012-06-01 00:00:00|MT_002   |[27.7382645803698,14.9359886201991,23.4708392603129,21.337126600284474,25.87126600284495,0.0,6.0] |24.8933143669986|
|2012-06-01 00:15:00|MT_002   |[25.6045519203414,16.3584637268848,22.7596017069701,24.00426742532005,25.893492176386914,0.0,6.0] |22.0483641536273|
|2012-06-01 00:30:00|MT_002   |[24.8933143669986,26.3157894736842,22.0483641536273,26.137980085348502,25.91571834992888,0.0,6.0] |22.0483641536273|
|2012-06-01 00:45:00|MT_002   |[22.0483641536273,27.7382645803698,22.0483641536273,25.071123755334277,25.9157183

In [25]:
import math
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def evaluate_predictions(
    df: DataFrame,
    label_col: str = "label",
    pred_cols: List[str] = None,
    group_col: str = None,
) -> DataFrame:
    """
    Evaluate one or multiple prediction columns against label.
    Returns a Spark DataFrame with metrics for each pred column.
    
    Metrics: n, MAE, RMSE
    If group_col is provided (e.g., 'client_id'), metrics are computed per group.
    """
    if pred_cols is None:
        pred_cols = []

    if not pred_cols:
        raise ValueError("pred_cols must include at least one prediction column name.")

    # create a long form: (group?, pred_name, label, pred)
    select_cols = []
    if group_col:
        select_cols.append(group_col)

    # stack: 'pred_naive', pred_naive, 'pred_seasonal', pred_seasonal, ...
    stack_expr = ", ".join([f"'{c}', `{c}`" for c in pred_cols])

    df_long = (
        df.select(
            *(select_cols + [F.col(label_col).alias("label")])
        )
        .selectExpr(
            *(select_cols + ["label",
                             f"stack({len(pred_cols)}, {stack_expr}) as (pred_name, pred)"])
        )
        .dropna(subset=["label", "pred"])
    )

    # metrics
    err = (F.col("pred") - F.col("label"))
    agg_exprs = [
        F.count(F.lit(1)).alias("n"),
        F.avg(F.abs(err)).alias("mae"),
        F.sqrt(F.avg(err * err)).alias("rmse"),
    ]

    if group_col:
        out = df_long.groupBy(group_col, "pred_name").agg(*agg_exprs).orderBy(group_col, "pred_name")
    else:
        out = df_long.groupBy("pred_name").agg(*agg_exprs).orderBy("pred_name")

    return out

### Implement the ML models

In [26]:
from pyspark.ml.evaluation import RegressionEvaluator

### Model A: Linear Regression (strong baseline)

In [27]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

lr_model = lr.fit(train_ml)

pred_lr = (
    lr_model
    .transform(test_ml)
    .withColumnRenamed("prediction", "pred_lr")
)

pred_lr.select("label", "pred_lr").show(5)

26/05/12 17:43:24 WARN Instrumentation: [51f637a7] regParam is zero, which might cause numerical instability and overfitting.


+----------------+------------------+
|           label|           pred_lr|
+----------------+------------------+
|24.8933143669986|27.585145306941772|
|22.0483641536273|24.750829636029312|
|22.0483641536273|  24.5024622428313|
|22.7596017069701|22.111324415304882|
|21.3371266002845| 22.24395133532034|
+----------------+------------------+
only showing top 5 rows


In [28]:
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_lr", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_lr", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_lr", metricName="r2")

rmse = rmse_eval.evaluate(pred_lr)
mae = mae_eval.evaluate(pred_lr)
r2 = r2_eval.evaluate(pred_lr)

print("Linear Regression Results")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

Linear Regression Results
RMSE: 2.2237
MAE : 1.0227
R2  : 0.9695


### Model B: Gradient Boost Tree

In [29]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_ml)

pred_gbt = (
    gbt_model
    .transform(test_ml)
    .withColumnRenamed("prediction", "pred_gbt")
)

pred_gbt.select("label", "pred_gbt").show(5)

+----------------+------------------+
|           label|          pred_gbt|
+----------------+------------------+
|24.8933143669986|25.255072623097316|
|22.0483641536273|20.802280562775348|
|22.0483641536273| 22.86762302937817|
|22.7596017069701| 21.92491413639818|
|21.3371266002845| 21.84062203585924|
+----------------+------------------+
only showing top 5 rows


In [31]:
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_gbt", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_gbt", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_gbt", metricName="r2")

rmse = rmse_eval.evaluate(pred_gbt)
mae = mae_eval.evaluate(pred_gbt)
r2 = r2_eval.evaluate(pred_gbt)

print("GBT Results")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

GBT Results
RMSE: 6.6525
MAE : 2.5312
R2  : 0.7272


### Model C: Random Forest

In [32]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
rf_model = rf.fit(train_ml)
pred_rf = rf_model.transform(test_ml).withColumnRenamed("prediction","pred_rf")

26/05/12 18:05:40 WARN DAGScheduler: Broadcasting large task binary with size 1075.9 KiB
26/05/12 18:05:42 WARN DAGScheduler: Broadcasting large task binary with size 1503.0 KiB
26/05/12 18:05:43 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


In [33]:
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="pred_rf", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="label", predictionCol="pred_rf", metricName="mae")
r2_eval   = RegressionEvaluator(labelCol="label", predictionCol="pred_rf", metricName="r2")

rmse = rmse_eval.evaluate(pred_rf)
mae  = mae_eval.evaluate(pred_rf)
r2   = r2_eval.evaluate(pred_rf)

print(f"LR test RMSE: {rmse:.4f}")
print(f"LR test MAE : {mae:.4f}")
print(f"LR test R2  : {r2:.4f}")

LR test RMSE: 6.2983
LR test MAE : 2.4889
LR test R2  : 0.7555
